<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML321ENSkillsNetwork817-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Collaborative Filtering based Recommender System using Non-negative Matrix Factorization**


Estimated time needed: **60** minutes


## **Introduction**

In the previous lab, we have performed KNN on user-item interaction matrix to estimate the rating of unknown items based on the aggregation of the user's K nearest neighbor's ratings. Finding nearest neighbors are based on similarity measurements among users or items with big similarity matrices. 


*The KNN algorithm is memory-based which means we need to keep all instances for prediction and maintain a big similarity matrix.* These can be infeasible if our user/item scale is large, for example, 1 million users will require a 1 million by 1 million similarity matrix, which is very hard to load into RAM for most computation environments.


### **Non-negative matrix factorization**


In this notebook, we will build a collaborative filtering recommender system using Non-negative Matrix Factorization (NMF). The input data is a user–course rating matrix, where each row is a user, each column is a course, and each value is the rating that user gave that course. Because most users rate only a small number of courses, this matrix is very sparse.

The main idea of NMF is to break this large sparse rating matrix into two smaller matrices: one that represents hidden user features and one that represents hidden course features. These are called **latent features**, because they are not given directly in the data but are learned from the rating patterns. After learning these two smaller matrices, we multiply them to reconstruct the original matrix and estimate ratings that are missing.

This notebook shows how NMF can be used as an alternative to KNN-based collaborative filtering. Unlike KNN, NMF does not rely on building a very large user-user or item-item similarity matrix, so it is often more scalable when the number of users or items becomes large.

#### **The main idea of NMF**
An example is shown below, suppose we have a user-item interaction matrix $A$ with 10000 users and 100 items (10000 x 100), and its element `(j, k)` represents the rating of item `k` from user `j`. Then we could decompose $A$ into two smaller and dense matrices $U$ (10000 x 16) and $I$ (16 x 100). for user matrix $U$, each row vector is a transformed **latent** feature vector of a user, and for the item matrix $I$, each column is a transformed latent feature vector of an item. 

Here the dimension 16 is a hyperparameter defines the size of the hidden user and item features, which means now the shape of transposed user feature vector and item feature vector is now 16 x 1.


*The magic here is when we multiply the row `j` of $U$ and column `k` of matrix $I$, we can get an estimation to the original rating $\hat{r}_{jk}$.*

For example, if we preform the dot product user ones  row vector in $U$ and item ones  column vector in $I$, we can get the rating estimation of user one to item one, which is the element (1, 1) in the original interaction matrix $I$. 


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/module_4/images/nmf.png)


Note $I$ is short for Items, and it is not an identity matrix.


#### **Matrix decomposition with least square algorithm**
Then how do we figure out the values in $U$ and $I$ exactly? Like many other machine learning processes, we could start by initializing the values of $U$ and $I$, then define the following distance or cost function to be minimized:


$$\sum_{r_{jk} \in {train}} \left(r_{jk} - \hat{r}_{jk} \right)^2,$$


where $\hat{r}_{ij}$ is the dot product of $u_j^T$ and $i_k$:


$$\hat{r}_{jk} = u_j^Ti_k$$


The cost function can be optimized using stochastic gradient descent (SGD) or other optimization algorithms, just like in training the weights in a logistic regression model (there are several additional steps so the matrices have no negative elements).

**Note:** ordinary NMF cannot directly take a matrix with true missing values (NaN) and decompose it. So recommender-system NMF does this instead: fit only on observed entries. it learns user and item latent factors by minimizing prediction error only on the observed ratings, and then uses those factors to estimate the missing ones.


## **Objectives**


* Perform NMF-based collaborative filtering on the user-item matrix


----


### **Load and exploring dataset**


Let's first load our dataset, i.e., the user-item (learn-course) interaction matrix


In [ ]:
import pandas as pd

In [ ]:
rating_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-ML0321EN-Coursera/labs/v2/module_3/ratings.csv"
rating_df = pd.read_csv(rating_url)

In [ ]:
rating_df.head()

The dataset contains three columns, `user id`, `item id`, and `the rating`. Note that this matrix is presented as the dense or vertical form, you may convert it using `pivot` to the original sparse matrix:


In [ ]:
rating_sparse_df = rating_df.pivot(index='user', columns='item', values='rating').fillna(0).reset_index().rename_axis(index=None, columns=None)
rating_sparse_df.head()

Next, you need to implement NMF-based collaborative filtering, and you may choose one of the two following implementation options: 
- The first one is to use `Surprise` which is a popular and easy-to-use Python recommendation system library. 
- The second way is to implement it with `numpy`, `pandas`, and `sklearn`. You may need to write a lot of low-level implementation code along the way.


### **Implementation Option 1: Use **Surprise** library (recommended)**


*Surprise* is a Python scikit library for recommender systems. It is simple and comprehensive to build and test different recommendation algorithms. First let's install it:


In [ ]:
#!pip install scikit-surprise

We import required classes and methods


In [ ]:
from surprise import NMF
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
from surprise import accuracy

In [ ]:
# Save the rating dataframe to a CSV file
rating_df.to_csv("course_ratings.csv", index=False)

# Read the course rating dataset with columns user item rating
reader = Reader(line_format='user item rating', sep=',', skip_lines=1, rating_scale=(2, 3))

# Load the dataset from the CSV file
course_dataset = Dataset.load_from_file("course_ratings.csv", reader=reader)

Now  we split the data into a train-set and test-set:


In [ ]:
trainset, testset = train_test_split(course_dataset, test_size=.3)

Then check how many users and items we can use to fit the KNN model:


In [ ]:
print(f"Total {trainset.n_users} users and {trainset.n_items} items in the trainingset")

##### **Perform NMF-based collaborative filtering on the course-interaction matrix**


_Fit a NMF model using the trainset and evaluate the results using the testset_ The code will be very similar to the KNN-based collaborative filtering, you just need to use the `NMF()` model.


In [ ]:
# - Define a NMF model NMF(verbose=True, random_state=123)
nmf_estimator = NMF(init_low=0.5, init_high = 5.0, n_factors=32)

# - Train the NMF on the trainset, and predict ratings for the testset
nmf_estimator.fit(trainset)
nmf_pred = nmf_estimator.test(testset)

# - Then compute RMSE
accuracy.rmse(nmf_pred)

In [ ]:
# try different hyperparamet combinations to see which one has the best performance
param_grid = {
    "n_factors": [5, 15, 30, 45],
    "n_epochs": {10, 20, 50},
}

gs = GridSearchCV(
    NMF,
    param_grid,
    measures=["rmse"],
    cv=3, 
    n_jobs=2 # use all CPUs
)

gs.fit(course_dataset)

# convert results to DataFrame
results_df = pd.DataFrame(gs.cv_results)
results_df = results_df[["param_n_factors", "param_n_epochs","mean_test_rmse"]]
results_df = results_df.sort_values("mean_test_rmse")

results_df

<details>
    <summary>Click here for Hints</summary>

* Create a model by calling `NMF()` class and set parameters to `init_low=0.5, init_high = 5.0, n_factors=32`. 
* Fit it  with `trainset` by using `model.fit(trainset)`.  
* Record predictions to the `testset`  by using `model.test(testset).
* Compute the accuracy by using `accuracy.rmse(predictions)`


To learn more detailed usages about _Surprise_ library, visit its website from [here](https://surprise.readthedocs.io/en/stable/getting_started.html?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML321ENSkillsNetwork817-2022-01-01)


### **Implementation Option 2: Use `numpy`, `pandas`, and `sklearn`**


If you do not prefer the one-stop Suprise solution, you may implement the KNN model using `numpy`, `pandas`, and possibly `sklearn`:


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, ParameterGrid
from sklearn.metrics import mean_squared_error

def masked_nmf(X, mask, n_comp, max_iter=100, seed=0):
    """
    Perform Non-Negative Matrix Factorization while ignoring missing values.
    
    Uses Multiplicative Update rules modified by a binary mask to ensure 
    the model only learns from observed ratings.
    
    Args:
        X: The user-item rating matrix (filled with 0s for missing values).
        mask: Binary matrix (1.0 for observed, 0.0 for missing).
        n_comp: Number of latent factors (rank of the approximation).
        max_iter: Number of optimization iterations.
        seed: Random seed for reproducible initialization of W and H.
    """
    rng = np.random.default_rng(seed)
    # Initialize W (User-Factor) and H (Factor-Item) with random positive values
    W = rng.random((X.shape[0], n_comp))
    H = rng.random((n_comp, X.shape[1]))
    
    for _ in range(max_iter):
        # Update H (Item Factors)
        # We mask both the actual data (mask * X) and the reconstruction (mask * WH)
        WH = W @ H
        H *= (W.T @ (mask * X)) / (W.T @ (mask * WH) + 1e-9)

        # Update W (User Factors) using the newly updated H
        WH = W @ H
        W *= ((mask * X) @ H.T) / ((mask * WH) @ H.T + 1e-9)
        
    return W @ H

# --- 1. Data Preparation ---
# Pivot long-format ratings into a wide User-Item matrix
ui_mat = rating_df.pivot(index="user", columns="item", values="rating")

# X_full: Dense matrix for calculation; obs: (row, col) coordinates of existing ratings
X_full = ui_mat.fillna(0).values
obs = np.argwhere(~ui_mat.isna().values)

# Get rating bounds for prediction clipping
r_min, r_max = rating_df["rating"].min(), rating_df["rating"].max()

# --- 2. Grid Search & Cross-Validation Setup ---
results = []
kf = KFold(n_splits=3, shuffle=True, random_state=0)
params = {"n_components": [5, 15, 30, 45], "max_iter": [10, 20, 50]}

# --- 3. Execution Loop ---
for p in ParameterGrid(params):
    fold_rmses = []
    
    # tr/te are indices of the 'obs' array, pointing to rating coordinates
    for tr, te in kf.split(obs):
        # Create a training mask: only 1.0 where ratings exist in the training fold
        mask = np.zeros_like(X_full)
        mask[obs[tr, 0], obs[tr, 1]] = 1.0
        
        # Fit model and generate the reconstructed matrix
        X_pred = masked_nmf(X_full, mask, p["n_components"], p["max_iter"])
        
        # Vectorized Evaluation: extract test positions from the prediction matrix
        r_te, c_te = obs[te].T
        y_true = ui_mat.values[r_te, c_te]
        
        # Clip predictions to valid rating range (e.g., 1-5) to improve accuracy
        y_pred = np.clip(X_pred[r_te, c_te], r_min, r_max)
        
        fold_rmses.append(np.sqrt(mean_squared_error(y_true, y_pred)))

    # Store hyperparameters and the average RMSE across folds
    results.append({**p, "mean_test_rmse": np.mean(fold_rmses)})

# --- 4. Reporting ---
# Display results sorted by performance (lowest RMSE first)
results_df = pd.DataFrame(results).sort_values("mean_test_rmse")
results_df

## **Summary**


In this lab, we have learned and practiced NMF-based collaborative filtering. The basic idea is to decompose the original user-item interaction matrix into two smaller and dense user and item matrices. Then, we have built the two matrices, we can easily estimate the unknown ratings via the dot product of specific row in user matrix and specific column in item matrix.


## **Authors**


[Yan Luo](https://www.linkedin.com/in/yan-luo-96288783/), Su Wu


Copyright © 2022 IBM Corporation. All rights reserved.
